# XLMROBERTA

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load datasets
train_df = pd.read_csv("/content/ag-train-binary-balanced.csv")
val_df = pd.read_csv("/content/ag-val-binary-balanced.csv")
test_df = pd.read_csv("/content/ag-test-binary-balanced.csv")

train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load XLM-RoBERTa
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Tokenization function
def tokenize(texts):
    return tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='tf'
    )

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ tf.data.Dataset objects
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ Wrap XLM-RoBERTa in a Keras Layer
class XLMRobertaWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.model = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build model using Functional API
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

# Last hidden states from XLM-RoBERTa
last_hidden_states = XLMRobertaWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

# Apply Attention
context_vector = AttentionLayer()(last_hidden_states)

# Dense Layers
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train model
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Predict + Classification Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["NoAG", "AG"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFXLMRobertaModel: ['lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing TFXLMRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFXLMRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFXLMRobertaModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFXLMRobertaModel for predictions without further training.


Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 135s 580ms/step - accuracy: 0.5318 - loss: 0.7520 - val_accuracy: 0.6955 - val_loss: 0.6509
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 70s 412ms/step - accuracy: 0.5754 - loss: 0.6735 - val_accuracy: 0.7131 - val_loss: 0.6305
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 69s 406ms/step - accuracy: 0.5947 - loss: 0.6663 - val_accuracy: 0.7244 - val_loss: 0.6134
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 68s 405ms/step - accuracy: 0.6309 - loss: 0.6460 - val_accuracy: 0.7308 - val_loss: 0.5978
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 69s 411ms/step - accuracy: 0.6274 - loss: 0.6392 - val_accuracy: 0.7388 - val_loss: 0.5837
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 69s 408ms/step - accuracy: 0.6704 - loss: 0.6108 - val_accuracy: 0.7564 - val_loss: 0.5703
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 68s 404ms/step - accuracy: 0.6887 - loss: 0.6062 - val_accuracy: 0.7564 - val_loss: 0.5588
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 68s 405ms/step - accuracy: 0.6947 - loss: 

# SENTIMENT

# MURIL

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load your dataset
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract text and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load MuRIL and tokenizer
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ MuRIL Wrapper Layer
class MuRILWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.muril = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.muril(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build the model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = MuRILWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train the model
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate and report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Positive", "Negative"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

tf_model.h5:   0%|          | 0.00/1.56G [00:00<?, ?B/s]

Some layers from the model checkpoint at google/muril-base-cased were not used when initializing TFBertModel: ['mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFBertModel were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['bert/pooler/dense/kernel:0', 'bert/pooler/dense/bias:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 52s 206ms/step - accuracy: 0.5311 - loss: 0.6928 - val_accuracy: 0.6042 - val_loss: 0.6921
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.5724 - loss: 0.6921 - val_accuracy: 0.6859 - val_loss: 0.6913
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 24s 142ms/step - accuracy: 0.6274 - loss: 0.6911 - val_accuracy: 0.6971 - val_loss: 0.6905
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.6391 - loss: 0.6906 - val_accuracy: 0.7067 - val_loss: 0.6897
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.6564 - loss: 0.6896 - val_accuracy: 0.7067 - val_loss: 0.6889
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.6603 - loss: 0.6888 - val_accuracy: 0.7067 - val_loss: 0.6881
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 137ms/step - accuracy: 0.6846 - loss: 0.6881 - val_accuracy: 0.7067 - val_loss: 0.6873
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 42s 144ms/step - accuracy: 0.6868 - loss: 0

# XLMROBERTA

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load your dataset
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract text and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load XLM-RoBERTa and tokenizer
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ XLM-RoBERTa Wrapper Layer
class XLMRobertaWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.roberta = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.roberta(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build the model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = XLMRobertaWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train the model
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate and report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Positive", "Negative"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFXLMRobertaModel: ['lm_head.layer_norm.bias', 'lm_head.dense.bias', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing TFXLMRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFXLMRobertaModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFXLMRobertaModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFXLMRobertaModel for predictions without further training.


Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 128s 550ms/step - accuracy: 0.4876 - loss: 0.8025 - val_accuracy: 0.5288 - val_loss: 0.6896
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 108s 409ms/step - accuracy: 0.5218 - loss: 0.7158 - val_accuracy: 0.5369 - val_loss: 0.6867
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 68s 402ms/step - accuracy: 0.5154 - loss: 0.7146 - val_accuracy: 0.5865 - val_loss: 0.6836
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 68s 400ms/step - accuracy: 0.5010 - loss: 0.7096 - val_accuracy: 0.5962 - val_loss: 0.6808
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 91s 455ms/step - accuracy: 0.5257 - loss: 0.7081 - val_accuracy: 0.6106 - val_loss: 0.6784
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 74s 406ms/step - accuracy: 0.5396 - loss: 0.6956 - val_accuracy: 0.6282 - val_loss: 0.6758
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 68s 401ms/step - accuracy: 0.5246 - loss: 0.7037 - val_accuracy: 0.6266 - val_loss: 0.6734
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 83s 405ms/step - accuracy: 0.5658 - loss:

# INDIC BERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load your dataset
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract text and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load IndicBERT and tokenizer
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ IndicBERT Wrapper Layer
class IndicBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.bert = TFAutoModel.from_pretrained(model_name, from_pt=True)

    def call(self, inputs):
        output = self.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build the model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = IndicBERTWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train the model
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate and report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Positive", "Negative"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFAlbertModel: ['predictions.decoder.weight', 'predictions.LayerNorm.bias', 'predictions.dense.bias', 'predictions.LayerNorm.weight', 'sop_classifier.classifier.bias', 'sop_classifier.classifier.weight', 'predictions.decoder.bias', 'predictions.bias', 'predictions.dense.weight']
- This IS expected if you are initializing TFAlbertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFAlbertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFAlbertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFAlbertModel

Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 53s 198ms/step - accuracy: 0.5106 - loss: 0.7160 - val_accuracy: 0.5737 - val_loss: 0.6841
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 25s 139ms/step - accuracy: 0.5348 - loss: 0.6908 - val_accuracy: 0.5978 - val_loss: 0.6773
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 24s 141ms/step - accuracy: 0.5610 - loss: 0.6823 - val_accuracy: 0.6202 - val_loss: 0.6723
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 135ms/step - accuracy: 0.5690 - loss: 0.6793 - val_accuracy: 0.6234 - val_loss: 0.6684
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.5678 - loss: 0.6767 - val_accuracy: 0.6170 - val_loss: 0.6653
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.5750 - loss: 0.6726 - val_accuracy: 0.6186 - val_loss: 0.6629
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.5829 - loss: 0.6690 - val_accuracy: 0.6202 - val_loss: 0.6607
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 136ms/step - accuracy: 0.5953 - loss: 0

# MBERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load your dataset
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract text and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load mBERT and tokenizer
mbert_model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(mbert_model_name)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ mBERT Wrapper Layer
class MBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.mbert = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.mbert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build the model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = MBERTWrapper(mbert_model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train the model
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate and report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Positive", "Negative"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 51s 206ms/step - accuracy: 0.5539 - loss: 0.6841 - val_accuracy: 0.6138 - val_loss: 0.6595
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 25s 139ms/step - accuracy: 0.6153 - loss: 0.6587 - val_accuracy: 0.6266 - val_loss: 0.6431
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.6510 - loss: 0.6411 - val_accuracy: 0.6587 - val_loss: 0.6316
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.6710 - loss: 0.6293 - val_accuracy: 0.6827 - val_loss: 0.6233
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.6640 - loss: 0.6170 - val_accuracy: 0.6795 - val_loss: 0.6170
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.6843 - loss: 0.6091 - val_accuracy: 0.6843 - val_loss: 0.6122
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.6893 - loss: 0.6034 - val_accuracy: 0.6859 - val_loss: 0.6085
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 42s 142ms/step - accuracy: 0.6854 - loss: 0

# BANGLA BERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report

# ✅ Load your dataset
train_df = pd.read_csv("/content/snt_balanced_train.csv")
val_df = pd.read_csv("/content/snt_balanced_dev.csv")
test_df = pd.read_csv("/content/snt_balanced_test.csv")

# ✅ Extract text and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load BanglaBERT and tokenizer
banglabert_model_name = "csebuetnlp/banglabert_large"
tokenizer = AutoTokenizer.from_pretrained(banglabert_model_name)

# ✅ Tokenize function
def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Convert to tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ BanglaBERT Wrapper Layer
class BanglaBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.banglabert = TFAutoModel.from_pretrained(model_name, from_pt=True)

    def call(self, inputs):
        output = self.banglabert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build the model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = BanglaBERTWrapper(banglabert_model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile the model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='binary_crossentropy',
              metrics=['accuracy'])

# ✅ Train the model
model.fit(train_dataset, validation_data=val_dataset, epochs=25)

# ✅ Evaluate and report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_binary = (preds.flatten() > 0.5).astype(int)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_binary, target_names=["Positive", "Negative"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFElectraModel: ['discriminator_predictions.dense_prediction.bias', 'discriminator_predictions.dense_prediction.weight', 'electra.embeddings.position_ids', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight']
- This IS expected if you are initializing TFElectraModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFElectraModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFElectraModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFElectraModel for predictions without further train

Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 129s 568ms/step - accuracy: 0.5536 - loss: 0.6916 - val_accuracy: 0.6010 - val_loss: 0.6665
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 78s 464ms/step - accuracy: 0.5928 - loss: 0.6736 - val_accuracy: 0.6058 - val_loss: 0.6605
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 81s 459ms/step - accuracy: 0.6067 - loss: 0.6657 - val_accuracy: 0.5978 - val_loss: 0.6573
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 458ms/step - accuracy: 0.6113 - loss: 0.6621 - val_accuracy: 0.6042 - val_loss: 0.6551
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 459ms/step - accuracy: 0.6068 - loss: 0.6539 - val_accuracy: 0.6154 - val_loss: 0.6528
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 457ms/step - accuracy: 0.6130 - loss: 0.6537 - val_accuracy: 0.6154 - val_loss: 0.6508
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 77s 455ms/step - accuracy: 0.6186 - loss: 0.6610 - val_accuracy: 0.6058 - val_loss: 0.6493
Epoch 8/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 77s 458ms/step - accuracy: 0.6277 - loss: 

# VIOLENCE

# MURIL

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer
model_name = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ MuRIL Wrapper
class MuRILWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.muril = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.muril(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = MuRILWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights to upweight label=2 (direct violence)
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluate
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["non-violence", "passive violence", "direct violence"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some layers from the model checkpoint at google/muril-base-cased were not used when initializing TFBertModel: ['mlm___cls']
- This IS expected if you are initializing TFBertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some layers of TFBertModel were not initialized from the model checkpoint at google/muril-base-cased and are newly initialized: ['bert/pooler/dense/kernel:0', 'bert/pooler/dense/bias:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class Weights: {0: np.float64(0.6479481641468683), 1: np.float64(0.9761388286334056), 2: np.float64(2.3136246786632393)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 52s 199ms/step - accuracy: 0.3093 - loss: 1.0998 - val_accuracy: 0.3990 - val_loss: 1.0978
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 25s 140ms/step - accuracy: 0.3567 - loss: 1.0992 - val_accuracy: 0.4567 - val_loss: 1.0971
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 138ms/step - accuracy: 0.4108 - loss: 1.0983 - val_accuracy: 0.5385 - val_loss: 1.0964
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 42s 142ms/step - accuracy: 0.4270 - loss: 1.0981 - val_accuracy: 0.5481 - val_loss: 1.0956
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 139ms/step - accuracy: 0.4506 - loss: 1.0970 - val_accuracy: 0.5529 - val_loss: 1.0948
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 142ms/step - accuracy: 0.4944 - loss: 1.0964 - val_accuracy: 0.5593 - val_loss: 1.0941
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 143ms/step - accuracy: 0.5076 - loss: 1.0956 - va

# INDIC BERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer for IndicBERT
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ IndicBERT Wrapper (instead of MuRIL)
class IndicBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.indicbert = TFAutoModel.from_pretrained(model_name, from_pt=True)

    def call(self, inputs):
        output = self.indicbert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = IndicBERTWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["non-violence", "passive violence", "direct violence"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFAlbertModel: ['predictions.decoder.weight', 'predictions.LayerNorm.bias', 'predictions.dense.bias', 'predictions.LayerNorm.weight', 'sop_classifier.classifier.bias', 'sop_classifier.classifier.weight', 'predictions.decoder.bias', 'predictions.bias', 'predictions.dense.weight']
- This IS expected if you are initializing TFAlbertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFAlbertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFAlbertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFAlbertModel

Class Weights: {0: np.float64(0.6479481641468683), 1: np.float64(0.9761388286334056), 2: np.float64(2.3136246786632393)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 51s 206ms/step - accuracy: 0.2606 - loss: 1.1209 - val_accuracy: 0.3237 - val_loss: 1.1087
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 137ms/step - accuracy: 0.3523 - loss: 1.1009 - val_accuracy: 0.3462 - val_loss: 1.0985
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 136ms/step - accuracy: 0.3431 - loss: 1.1009 - val_accuracy: 0.3798 - val_loss: 1.0903
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 137ms/step - accuracy: 0.3938 - loss: 1.0767 - val_accuracy: 0.3814 - val_loss: 1.0840
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 141ms/step - accuracy: 0.3897 - loss: 1.0727 - val_accuracy: 0.3894 - val_loss: 1.0781
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 23s 138ms/step - accuracy: 0.4056 - loss: 1.0718 - val_accuracy: 0.3974 - val_loss: 1.0729
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 24s 140ms/step - accuracy: 0.4334 - loss: 1.0598 - va

MBERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer for mBERT
mbert_model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(mbert_model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ mBERT Wrapper
class mBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.bert = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.bert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = mBERTWrapper(mbert_model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["non-violence", "passive violence", "direct violence"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions w

Class Weights: {0: np.float64(0.6479481641468683), 1: np.float64(0.9761388286334056), 2: np.float64(2.3136246786632393)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 50s 198ms/step - accuracy: 0.3373 - loss: 1.1225 - val_accuracy: 0.3478 - val_loss: 1.0999
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 26s 142ms/step - accuracy: 0.3669 - loss: 1.0884 - val_accuracy: 0.4006 - val_loss: 1.0739
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 139ms/step - accuracy: 0.4191 - loss: 1.0555 - val_accuracy: 0.4519 - val_loss: 1.0508
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 42s 147ms/step - accuracy: 0.4905 - loss: 1.0251 - val_accuracy: 0.4952 - val_loss: 1.0306
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 40s 140ms/step - accuracy: 0.5223 - loss: 1.0034 - val_accuracy: 0.5080 - val_loss: 1.0123
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 42s 143ms/step - accuracy: 0.5235 - loss: 0.9878 - val_accuracy: 0.5240 - val_loss: 0.9965
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 41s 141ms/step - accuracy: 0.5548 - loss: 0.9684 - va

XLMROBERTA

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Load XLM-RoBERTa tokenizer
model_name = "xlm-roberta-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Custom Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ XLM-R Wrapper
class XLMRobertaWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.model = TFAutoModel.from_pretrained(model_name)

    def call(self, inputs):
        output = self.model(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Build Model
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = XLMRobertaWrapper(model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute Class Weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train Model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["non-violence", "passive violence", "direct violence"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFXLMRobertaModel: ['lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.bias']
- This IS expected if you are initializing TFXLMRobertaModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFXLMRobertaModel from a PyTorc

Class Weights: {0: np.float64(0.6479481641468683), 1: np.float64(0.9761388286334056), 2: np.float64(2.3136246786632393)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 136s 588ms/step - accuracy: 0.4271 - loss: 1.2856 - val_accuracy: 0.3510 - val_loss: 1.1044
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 112s 480ms/step - accuracy: 0.3337 - loss: 1.1593 - val_accuracy: 0.3766 - val_loss: 1.0972
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 71s 414ms/step - accuracy: 0.3421 - loss: 1.1405 - val_accuracy: 0.3990 - val_loss: 1.0915
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 414ms/step - accuracy: 0.3525 - loss: 1.1362 - val_accuracy: 0.4215 - val_loss: 1.0869
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 76s 451ms/step - accuracy: 0.3420 - loss: 1.1336 - val_accuracy: 0.4503 - val_loss: 1.0823
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 75s 413ms/step - accuracy: 0.3730 - loss: 1.1202 - val_accuracy: 0.4311 - val_loss: 1.0804
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 414ms/step - accuracy: 0.3908 - loss: 1.1087 - 

BANGLA BERT

In [ ]:
import pandas as pd
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# ✅ Load dataset
train_df = pd.read_csv("/content/vio-train-balanced.csv")
val_df = pd.read_csv("/content/vio-dev-balanced.csv")
test_df = pd.read_csv("/content/vio-test-balanced.csv")

# ✅ Extract texts and labels
train_texts = train_df["text"].tolist()
train_labels = train_df["label"].tolist()
val_texts = val_df["text"].tolist()
val_labels = val_df["label"].tolist()
test_texts = test_df["text"].tolist()
test_labels = test_df["label"].tolist()

# ✅ Tokenizer for BanglaBERT
banglabert_model_name = "csebuetnlp/banglabert_large"
tokenizer = AutoTokenizer.from_pretrained(banglabert_model_name)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=128, return_tensors='tf')

train_encodings = tokenize(train_texts)
val_encodings = tokenize(val_texts)
test_encodings = tokenize(test_texts)

# ✅ Prepare tf.data.Dataset
train_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': train_encodings['input_ids'],
    'attention_mask': train_encodings['attention_mask']
}, train_labels)).batch(16)

val_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': val_encodings['input_ids'],
    'attention_mask': val_encodings['attention_mask']
}, val_labels)).batch(16)

test_dataset = tf.data.Dataset.from_tensor_slices(({
    'input_ids': test_encodings['input_ids'],
    'attention_mask': test_encodings['attention_mask']
}, test_labels)).batch(16)

# ✅ Attention Layer
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self):
        super(AttentionLayer, self).__init__()

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)

    def call(self, hidden_states):
        score = tf.nn.tanh(tf.matmul(hidden_states, self.W) + self.b)
        attention_weights = tf.nn.softmax(score, axis=1)
        context_vector = attention_weights * hidden_states
        context_vector = tf.reduce_sum(context_vector, axis=1)
        return context_vector

# ✅ BanglaBERT Wrapper
class BanglaBERTWrapper(tf.keras.layers.Layer):
    def __init__(self, model_name):
        super().__init__()
        self.banglabert = TFAutoModel.from_pretrained(model_name, from_pt=True)

    def call(self, inputs):
        output = self.banglabert(
            input_ids=inputs['input_ids'],
            attention_mask=inputs['attention_mask']
        )
        return output.last_hidden_state

# ✅ Model Architecture
input_ids = tf.keras.Input(shape=(128,), dtype=tf.int32, name="input_ids")
attention_mask = tf.keras.Input(shape=(128,), dtype=tf.int32, name="attention_mask")

last_hidden_states = BanglaBERTWrapper(banglabert_model_name)({
    'input_ids': input_ids,
    'attention_mask': attention_mask
})

context_vector = AttentionLayer()(last_hidden_states)
x = tf.keras.layers.Dense(128, activation='relu')(context_vector)
x = tf.keras.layers.Dropout(0.1)(x)
output = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=[input_ids, attention_mask], outputs=output)

# ✅ Compile model
model.compile(optimizer=tf.keras.optimizers.Adam(2e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# ✅ Compute class weights
class_weights = compute_class_weight(class_weight='balanced',
                                     classes=np.unique(train_labels),
                                     y=train_labels)
class_weights_dict = {i: weight for i, weight in enumerate(class_weights)}
print("Class Weights:", class_weights_dict)

# ✅ Train model
model.fit(train_dataset,
          validation_data=val_dataset,
          epochs=25,
          class_weight=class_weights_dict)

# ✅ Evaluation Report
def report(dataset, true_labels, name="Set"):
    preds = model.predict(dataset)
    preds_class = np.argmax(preds, axis=1)
    print(f"\n📊 Classification Report for {name}:")
    print(classification_report(true_labels, preds_class, target_names=["non-violence", "passive violence", "direct violence"]))

report(train_dataset, train_labels, name="Train Set")
report(val_dataset, val_labels, name="Validation Set")
report(test_dataset, test_labels, name="Test Set")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


pytorch_model.bin:   0%|          | 0.00/1.35G [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFElectraModel: ['discriminator_predictions.dense_prediction.weight', 'discriminator_predictions.dense_prediction.bias', 'electra.embeddings.position_ids', 'discriminator_predictions.dense.bias', 'discriminator_predictions.dense.weight']
- This IS expected if you are initializing TFElectraModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFElectraModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFElectraModel were initialized from the Py

Class Weights: {0: np.float64(0.6479481641468683), 1: np.float64(0.9761388286334056), 2: np.float64(2.3136246786632393)}
Epoch 1/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 123s 547ms/step - accuracy: 0.3241 - loss: 1.1993 - val_accuracy: 0.3830 - val_loss: 1.1022
Epoch 2/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 118s 458ms/step - accuracy: 0.3538 - loss: 1.1219 - val_accuracy: 0.3878 - val_loss: 1.0927
Epoch 3/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 457ms/step - accuracy: 0.3514 - loss: 1.1240 - val_accuracy: 0.4071 - val_loss: 1.0871
Epoch 4/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 456ms/step - accuracy: 0.3647 - loss: 1.0997 - val_accuracy: 0.4151 - val_loss: 1.0834
Epoch 5/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 77s 455ms/step - accuracy: 0.3692 - loss: 1.1085 - val_accuracy: 0.4135 - val_loss: 1.0800
Epoch 6/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 456ms/step - accuracy: 0.3929 - loss: 1.0888 - val_accuracy: 0.4135 - val_loss: 1.0769
Epoch 7/25
169/169 ━━━━━━━━━━━━━━━━━━━━ 82s 456ms/step - accuracy: 0.3939 - loss: 1.0849 - 